<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/start_war_char_species_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "star_wars_character_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "ulrikthygepedersen/star-wars-characters",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_15777/1550875468.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'star-wars-characters' dataset.
First 5 records:              name  height   mass hair_color   skin_color eye_color  \
0  Luke Skywalker   172.0   77.0      blond         fair      blue   
1           C-3PO   167.0   75.0        NaN         gold    yellow   
2           R2-D2    96.0   32.0        NaN  white, blue       red   
3     Darth Vader   202.0  136.0       none        white    yellow   
4     Leia Organa   150.0   49.0      brown        light     brown   

   birth_year     sex     gender homeworld species  \
0        19.0    male  masculine  Tatooine   Human   
1       112.0    none  masculine  Tatooine   Droid   
2        33.0    none  masculine     Naboo   Droid   
3        41.9    male  masculine  Tatooine   Human   
4        19.0  female   feminine  Alderaan   Human   

                                               films  \
0  The Empire Strikes Back, Revenge of the Sith, ...   
1  The Empire Strikes Back, Attack of the Clones,.

In [53]:
df

,name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
0,Luke Skywalker,172.0,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith, ...","Snowspeeder, Imperial Speeder Bike","X-wing, Imperial shuttle"
1,C-3PO,167.0,75.0,NaN,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones,...",NaN,NaN
2,R2-D2,96.0,32.0,NaN,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones,...",NaN,NaN
3,Darth Vader,202.0,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith, ...",NaN,TIE Advanced x1
4,Leia Organa,150.0,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith, ...",Imperial Speeder Bike,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,Rey,NaN,NaN,brown,light,hazel,NaN,female,feminine,NaN,Human,The Force Awakens,NaN,NaN
83,Poe Dameron,NaN,NaN,brown,light,brown,NaN,male,masculine,NaN,Human,The Force Awakens,NaN,T-70 X-wing fighter
84,BB8,NaN,NaN,none,none,black,NaN,none,masculine,NaN,Droid,The Force Awakens,NaN,NaN
85,Captain Phasma,NaN,NaN,unknown,unknown,unknown,NaN,NaN,NaN,NaN,NaN,The Force Awakens,NaN,NaN


In [54]:
df['height'] = df['height'].fillna(df['height'].median())
df['mass'] = df['mass'].fillna(df['mass'].median())

In [55]:
df['hair_color'] = df['hair_color'].fillna("Unknown")
df['gender'] = df['gender'].fillna("Unknown")
df['sex'] = df['sex'].fillna("Unknown")
df['species'] = df['species'].fillna("Unknown")
df['homeworld'] = df['homeworld'].fillna("Unknown")

In [56]:
df = df.drop(['birth_year','vehicles','starships','films'],axis=1)

In [57]:
df

,name,height,mass,hair_color,skin_color,eye_color,sex,gender,homeworld,species
0,Luke Skywalker,172.0,77.0,blond,fair,blue,male,masculine,Tatooine,Human
1,C-3PO,167.0,75.0,Unknown,gold,yellow,none,masculine,Tatooine,Droid
2,R2-D2,96.0,32.0,Unknown,"white, blue",red,none,masculine,Naboo,Droid
3,Darth Vader,202.0,136.0,none,white,yellow,male,masculine,Tatooine,Human
4,Leia Organa,150.0,49.0,brown,light,brown,female,feminine,Alderaan,Human
...,...,...,...,...,...,...,...,...,...,...
82,Rey,180.0,79.0,brown,light,hazel,female,feminine,Unknown,Human
83,Poe Dameron,180.0,79.0,brown,light,brown,male,masculine,Unknown,Human
84,BB8,180.0,79.0,none,none,black,none,masculine,Unknown,Droid
85,Captain Phasma,180.0,79.0,unknown,unknown,unknown,Unknown,Unknown,Unknown,Unknown


In [58]:
df = df.drop('name',axis=1)

In [59]:
x = df.drop(['species'],axis=1)
y = df['species']

In [60]:
numeric_cols = [
    "height",
    "mass"
]

In [61]:
categorical_cols = [
    "hair_color",
    "skin_color",
    "eye_color",
    "sex",
    "gender",
    "homeworld",
]

In [62]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


In [63]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(), categorical_cols)]
)

In [64]:
x = preprocessor.fit_transform(x)

In [65]:
x = x.toarray()

In [66]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [67]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

In [68]:
class StartWarDataset(Dataset):
    def __init__(self,x,y):
      self.x = torch.tensor(x,dtype=torch.float32)
      self.y = torch.tensor(y,dtype=torch.long)

    def __len__(self):
      return len(self.x)

    def __getitem__(self,idx):
      return self.x[idx],self.y[idx]

In [69]:
model = nn.Sequential(
    nn.Linear(x.shape[1],128),
    nn.ReLU(),
    nn.Linear(128,64),
    nn.ReLU(),
    nn.Linear(64,len(le.classes_))
)

In [70]:
optimizer = optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()


In [71]:
from sklearn.model_selection import train_test_split

In [72]:
train_x, test_x, train_y, test_y = train_test_split(x,y,test_size=0.2,random_state=42)

In [73]:
train_dataset = StartWarDataset(train_x,train_y)
test_dataset = StartWarDataset(test_x, test_y)

In [74]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [75]:
epochs=100

for epoch in range(epochs):
  model.train()
  total_loss = 0
  for batch_x,batch_y in train_loader:
    optimizer.zero_grad()
    output = model(batch_x)
    loss = criterion(output,batch_y)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  if epoch % 10 == 0:
    print(f"Epoch {epoch}, Loss: {total_loss / len(train_loader)}")




Epoch 0, Loss: 3.5999598503112793
Epoch 10, Loss: 3.092742840449015
Epoch 20, Loss: 2.1301934719085693
Epoch 30, Loss: 1.5547901391983032
Epoch 40, Loss: 1.5905603965123494
Epoch 50, Loss: 0.9548214475313822
Epoch 60, Loss: 0.6443843444188436
Epoch 70, Loss: 0.34652552505334216
Epoch 80, Loss: 0.2653098752101262
Epoch 90, Loss: 0.09579643482963245


In [76]:
model.eval()
with torch.no_grad():
  correct = 0
  total = 0
  for batch_x,batch_y in test_loader:
    output = model(batch_x)
    _,predicted = torch.max(output.data,1)
    total += batch_y.size(0)
    correct += (predicted == batch_y).sum().item()

  print(f"Accuracy: {100 * correct / total}%")





Accuracy: 66.66666666666667%
